# ManualBot — Semana 2: Chunking, Embeddings, Banco Vetorial e Busca Semântica

Este notebook contém os experimentos e a validação completa da **Semana 2** do projeto **ManualBot (ESP32)**.

## Objetivos da Semana 2:
1. **Implementar Chunking**: Divisão em blocos de 800 caracteres com overlap de 100 caracteres usando `RecursiveCharacterTextSplitter`.
2. **Gerar Embeddings**: Utilização do modelo open-source `sentence-transformers/all-MiniLM-L6-v2`.
3. **Construir o Banco Vetorial**: Banco de dados vetorial persistente com **ChromaDB**.
4. **Validar a Busca Semântica**: Consulta e recuperação de trechos relevantes dos manuais do ESP32.

--- 
## 1. Importação das Bibliotecas

In [ ]:
import os
import sys
from pathlib import Path

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print("Bibliotecas importadas com sucesso!")

--- 
## 2. Carregamento dos PDFs e Configuração de Chunking

In [ ]:
PASTA_DOCUMENTOS = Path("../docs/PDFs-Instrucoes")
PASTA_CHROMA = Path("../data/chroma_db")
MODELO_EMBEDDING = "sentence-transformers/all-MiniLM-L6-v2"

print(f"Carregando PDFs de: {PASTA_DOCUMENTOS.resolve()}")
loader = PyPDFDirectoryLoader(str(PASTA_DOCUMENTOS))
documentos = loader.load()
print(f"Total de páginas carregadas: {len(documentos)}")

# Estratégia de Chunking (800 chars / 100 overlap)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_documents(documentos)
print(f"Total de chunks gerados: {len(chunks)}")

--- 
## 3. Geração de Embeddings e Construção do Banco Vetorial ChromaDB

In [ ]:
print(f"Inicializando modelo de embeddings: {MODELO_EMBEDDING}")
embeddings = HuggingFaceEmbeddings(
    model_name=MODELO_EMBEDDING,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

PASTA_CHROMA.mkdir(parents=True, exist_ok=True)

print("Gerando vetores e salvando no ChromaDB...")
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=str(PASTA_CHROMA)
)
print("Banco vetorial criado e salvo com sucesso em data/chroma_db!")

--- 
## 4. Validação da Busca Semântica

In [ ]:
perguntas_teste = [
    "What is the operating voltage range for the ESP32?",
    "How many GPIO pins does the ESP32 have?",
    "What are the Wi-Fi protocols supported by the ESP32?",
    "How does deep sleep mode work on ESP32?"
]

for i, pergunta in enumerate(perguntas_teste, 1):
    print(f"\n[Pergunta {i}]: {pergunta}")
    print("=" * 60)
    resultados = vector_store.similarity_search_with_score(pergunta, k=3)
    for j, (doc, score) in enumerate(resultados, 1):
        fonte = Path(doc.metadata.get('source', 'Desconhecido')).name
        pagina = doc.metadata.get('page', 0) + 1
        print(f"  Resultado {j} (Score Distância: {score:.4f}):")
        print(f"  Fonte: {fonte} | Página: {pagina}")
        preview = doc.page_content.replace('\n', ' ').strip()[:180]
        print(f"  Trecho: \"{preview}...\"")
        print("  " + "-" * 45)